# Run the fire-calendar pipeline in Google Colab

Runs the scripts in `pipeline/` (README Section 2.2) on a Colab machine. Raw downloads are large, so they are stored on your Google Drive and survive Colab restarts.

**The repo is private**, so before running cell 3, upload a GitHub "Download ZIP" of the repo to your Drive as `nasa-fire-calendar/repo.zip`.

**Status: written but not yet run.** The `pipeline/` scripts do not exist yet (README Section 10, steps S09 onward), so cells 6 to 8 will say "nothing to run" until they do. Run cells top to bottom. When you finish, copy the summary printed by the last cell into `docs/HANDOFF.md` (README Section 11.3), because a Colab session is wiped when it ends.

**Before you start (once):** in Colab, open the key icon (Secrets) in the left bar, add a secret named `FIRMS_MAP_KEY` with your FIRMS key, and turn on "Notebook access". Never paste the key into a cell.

In [ ]:
# 1. Settings. Change BRANCH if the step you are doing lives on an unmerged branch (see the newest docs/HANDOFF.md entry).
REPO_URL = "https://github.com/shrihanchada-rgb/NASA-Space-Apps---YASH-AND-SHRIHAN"
BRANCH = "main"
REPO_DIR = "/content/repo"
DRIVE_DIR = "/content/drive/MyDrive/nasa-fire-calendar"   # raw data and results are kept here
# Which numbered scripts to run, in order, e.g. ["01", "02", "03"]. Leave empty to only set up.
STEPS = []

In [ ]:
# 2. Connect Google Drive (a pop-up asks for permission).
from google.colab import drive
drive.mount("/content/drive")
import os
os.makedirs(DRIVE_DIR, exist_ok=True)
print("Drive folder:", DRIVE_DIR)

In [ ]:
# 3. Get the code. The repo is PRIVATE, so Colab cannot clone it without a login.
# Easiest fix, no login or token: on GitHub click Code > Download ZIP, upload that file to your Drive folder
# (My Drive/nasa-fire-calendar/) and name it repo.zip. Re-upload a fresh zip whenever the code changes.
import subprocess, os, shutil, zipfile
def sh(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, cwd=cwd, text=True, capture_output=True)
    print(r.stdout[-3000:] + r.stderr[-3000:])
    return r.returncode

ZIP_PATH = f"{DRIVE_DIR}/repo.zip"
if os.path.exists(ZIP_PATH):
    print("Using", ZIP_PATH)
    tmp = "/content/_repo_unzip"
    shutil.rmtree(tmp, ignore_errors=True)
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(tmp)
    tops = [d for d in os.listdir(tmp) if os.path.isdir(os.path.join(tmp, d))]
    assert len(tops) == 1, f"Expected one top folder in the zip, found {tops}"
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    shutil.move(os.path.join(tmp, tops[0]), REPO_DIR)
elif os.path.isdir(REPO_DIR + "/.git"):
    sh(f"git fetch origin {BRANCH} && git checkout {BRANCH} && git pull origin {BRANCH}", cwd=REPO_DIR)
else:
    print("No repo.zip in Drive. Trying git clone, which only works if the repo is public.")
    if sh(f"git clone --depth 1 -b {BRANCH} {REPO_URL} {REPO_DIR}") != 0:
        raise SystemExit("Clone failed. The repo is private: upload repo.zip to " + DRIVE_DIR + " and re-run this cell.")
print("Code is in", REPO_DIR, "->", sorted(os.listdir(REPO_DIR))[:12])


In [ ]:
# 4. Install Python packages. Uses requirements.txt once the pipeline has one; until then a default list.
if os.path.exists(f"{REPO_DIR}/requirements.txt"):
    sh("pip install -q -r requirements.txt", cwd=REPO_DIR)
else:
    sh("pip install -q pandas pyarrow geopandas duckdb requests pytest")

In [ ]:
# 5. Load the FIRMS key from Colab Secrets (it is never printed) and keep big folders on Drive.
from google.colab import userdata
try:
    os.environ["FIRMS_MAP_KEY"] = userdata.get("FIRMS_MAP_KEY")
    print("FIRMS_MAP_KEY loaded:", bool(os.environ["FIRMS_MAP_KEY"]))
except Exception as e:
    print("FIRMS_MAP_KEY not available. Add it in Colab Secrets (key icon) and allow notebook access.", type(e).__name__)

# data/raw and data/interim are gitignored. Point them at Drive so downloads survive a restart and the scripts can resume.
for name in ("raw", "interim"):
    target = f"{DRIVE_DIR}/{name}"
    link = f"{REPO_DIR}/data/{name}"
    os.makedirs(target, exist_ok=True)
    os.makedirs(f"{REPO_DIR}/data", exist_ok=True)
    if not os.path.islink(link):
        if os.path.exists(link):
            sh(f"cp -rn {link}/. {target}/ && rm -rf {link}")
        os.symlink(target, link)
    print(link, "->", os.readlink(link))

In [ ]:
# 6. Run the numbered pipeline scripts you chose in STEPS, stopping at the first failure.
import glob
results = []
if not STEPS:
    print("STEPS is empty: nothing to run. Set STEPS in cell 1 (for example ['01']) and re-run from cell 1.")
for step in STEPS:
    matches = sorted(glob.glob(f"{REPO_DIR}/pipeline/{step}_*.py"))
    if not matches:
        print(f"No script found for step {step} in pipeline/. It has not been written yet.")
        results.append((step, "missing"))
        break
    script = matches[0]
    print("Running", os.path.basename(script))
    code_ = sh(f"python {script}", cwd=REPO_DIR)
    results.append((os.path.basename(script), "ok" if code_ == 0 else f"failed (exit {code_})"))
    if code_ != 0:
        break
print(results)

In [ ]:
# 7. Run the tests (tests must not download data; README Section 11.2 rules).
if os.path.isdir(f"{REPO_DIR}/pipeline/tests"):
    sh("pytest -q pipeline/tests", cwd=REPO_DIR)
else:
    print("No pipeline/tests yet.")

In [ ]:
# 8. Save results to Drive and print a summary to paste into docs/HANDOFF.md.
processed = f"{REPO_DIR}/data/processed"
if os.path.isdir(processed):
    sh(f"mkdir -p {DRIVE_DIR}/processed && cp -r {processed}/. {DRIVE_DIR}/processed/")
print("Steps run:", results)
for folder in ("raw", "interim", "processed"):
    path = f"{DRIVE_DIR}/{folder}"
    files = [os.path.join(d, f) for d, _, fs in os.walk(path) for f in fs] if os.path.isdir(path) else []
    size_mb = sum(os.path.getsize(f) for f in files) / 1e6
    print(f"{folder}: {len(files)} files, {size_mb:.1f} MB in {path}")
if os.path.isdir(REPO_DIR + "/.git"):
    print("Commit:"); sh("git rev-parse HEAD", cwd=REPO_DIR)
else:
    print("Code came from repo.zip (no git history). Note the date you downloaded the zip.")
print("Paste the lines above into your docs/HANDOFF.md entry, with the exact STEPS you used.")